In [1]:
import os
import pyreadstat
import pandas as pd
import numpy as np

school_file_path = r'C:\Users\ASUS\pisa_project\data\CY09_MS_SCH_PUF\SCH_QQQ_SAS\CY08MSP_SCH_QQQ.SAS7BDAT'
student_file_path = r'C:\Users\ASUS\pisa_project\data\STU_QQQ_SAS\CY08MSP_STU_QQQ.SAS7BDAT'

assert os.path.exists(school_file_path), f"School file not found at: {school_file_path}"
assert os.path.exists(student_file_path), f"Student file not found at: {student_file_path}"

print("All dependencies imported and PISA 2022 raw file paths verified successfully!")

All dependencies imported and PISA 2022 raw file paths verified successfully!


In [2]:
# Define targeted school-level infrastructure and staffing indicators
school_target_cols = [
    'CNT', 'CNTSCHID',         # Composite primary keys (Country code and School ID)
    'CLSIZE',                  # Average class size
    'STRATIO_Q',               # Student-to-teacher ratio (validated 2022 variable)
    'PROATCE',                 # Proportion of fully certified teachers
    'EDUSHORT',                # Educational material / infrastructure shortages
    'STAFFSHORT',              # Shortage of instructional staff
    'SCHLTYPE'                 # Institutional school governance (public vs. private)
]

# Read only necessary variables to minimize memory allocation
df_school, school_meta = pyreadstat.read_sas7bdat(
    school_file_path, 
    usecols=school_target_cols
)

print(f"School data loaded: {df_school.shape[0]} schools across {df_school.shape[1]} attributes.")
df_school.head(3)

School data loaded: 21629 schools across 7 attributes.


,CNT,CNTSCHID,SCHLTYPE,PROATCE,STAFFSHORT,EDUSHORT,CLSIZE
0,ALB,800001.0,3.0,1.0,-0.2968,1.2048,33.0
1,ALB,800002.0,3.0,1.0,-1.4551,2.9595,13.0
2,ALB,800003.0,1.0,1.0,0.2833,0.6299,13.0


In [3]:
# Preserve all 10 plausible values to accurately capture cognitive measurement variance
student_target_cols = [
    'CNT', 'CNTSCHID',         # Composite relational keys matching the school questionnaire
    'ESCS',                    # Index of economic, social, and cultural status
    'W_FSTUWT',                # Final trimmed student sampling weight
    'PV1MATH', 'PV2MATH',      # 10 Plausible Values for mathematics literacy
    'PV3MATH', 'PV4MATH',
    'PV5MATH', 'PV6MATH',
    'PV7MATH', 'PV8MATH',
    'PV9MATH', 'PV10MATH'
]

# Stream only target attributes from the uncompressed student dataset
df_student, student_meta = pyreadstat.read_sas7bdat(
    student_file_path, 
    usecols=student_target_cols
)

print(f"Student data loaded: {df_student.shape[0]} students across {df_student.shape[1]} attributes.")
df_student.head(3)

Student data loaded: 613744 students across 14 attributes.


,CNT,CNTSCHID,ESCS,W_FSTUWT,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,PV6MATH,PV7MATH,PV8MATH,PV9MATH,PV10MATH
0,ALB,800282.0,1.1112,3.15874,179.583,201.334,222.248,194.943,222.755,213.100,221.021,293.710,226.912,254.745
1,ALB,800115.0,-3.0507,4.34524,308.247,304.444,316.129,234.353,323.448,303.435,333.178,288.888,343.157,329.659
2,ALB,800242.0,-0.1867,7.83860,267.514,343.743,327.157,290.771,316.227,309.451,323.337,307.508,356.244,295.401


In [4]:
# Select comparable European educational systems for comparative benchmarking
target_countries = ['DEU', 'FRA', 'POL', 'EST', 'ITA', 'ESP', 'UKR']

# Filter both datasets before executing the join to optimize computational speed
df_school_sub = df_school[df_school['CNT'].isin(target_countries)].copy()
df_student_sub = df_student[df_student['CNT'].isin(target_countries)].copy()

# Harmonize key data types to prevent type mismatch exceptions during merge
df_student_sub['CNTSCHID'] = df_student_sub['CNTSCHID'].astype(float)
df_school_sub['CNTSCHID'] = df_school_sub['CNTSCHID'].astype(float)
df_student_sub['CNT'] = df_student_sub['CNT'].astype(str)
df_school_sub['CNT'] = df_school_sub['CNT'].astype(str)

# Perform inner merge to retain students linked to completed school records
merged_df = pd.merge(
    df_student_sub,
    df_school_sub,
    on=['CNT', 'CNTSCHID'],
    how='inner'
)

print(f"Merged dimensions: {merged_df.shape[0]} observations, {merged_df.shape[1]} features.")

Merged dimensions: 66641 observations, 19 features.


In [5]:
# 1. Drop records lacking core sampling weights or baseline cognitive values
merged_df = merged_df.dropna(subset=['W_FSTUWT', 'PV1MATH'])

# 2. Select numeric school and socioeconomic indicators requiring imputation
impute_candidates = ['ESCS', 'STRATIO_Q', 'PROATCE', 'EDUSHORT', 'STAFFSHORT', 'CLSIZE']
existing_impute_cols = [col for col in impute_candidates if col in merged_df.columns]

# 3. Apply country-level median imputation to maintain between-country variance structures
for col in existing_impute_cols:
    merged_df[col] = merged_df.groupby('CNT')[col].transform(
        lambda group: group.fillna(group.median())
    )

# 4. Remove residual records where national medians could not be computed
required_analysis_cols = ['ESCS', 'PROATCE', 'EDUSHORT']
if 'STRATIO_Q' in merged_df.columns:
    required_analysis_cols.append('STRATIO_Q')

merged_df = merged_df.dropna(subset=required_analysis_cols)

print(f"Cleaned sample ready for downstream modeling: {len(merged_df)} observations.")
print("Retained feature set:\n", list(merged_df.columns))

Cleaned sample ready for downstream modeling: 66641 observations.
Retained feature set:
 ['CNT', 'CNTSCHID', 'ESCS', 'W_FSTUWT', 'PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH', 'PV6MATH', 'PV7MATH', 'PV8MATH', 'PV9MATH', 'PV10MATH', 'SCHLTYPE', 'PROATCE', 'STAFFSHORT', 'EDUSHORT', 'CLSIZE']


In [6]:
# Create dedicated destination directory for processed datasets
output_directory = r'C:\Users\ASUS\pisa_project\data\processed'
os.makedirs(output_directory, exist_ok=True)

# Define output path
processed_file_path = os.path.join(output_directory, 'pisa_cleaned_dataset.parquet')

# Export to Parquet format to ensure rapid I/O throughput and compact disk storage
merged_df.to_parquet(processed_file_path, index=False)

print(f"Pipeline executed successfully.")
print(f"Cleaned dataset saved at: {processed_file_path}")
print(f"Final shape: {merged_df.shape[0]} rows by {merged_df.shape[1]} columns.")
merged_df.head()

Pipeline executed successfully.
Cleaned dataset saved at: C:\Users\ASUS\pisa_project\data\processed\pisa_cleaned_dataset.parquet
Final shape: 66641 rows by 19 columns.


,CNT,CNTSCHID,ESCS,W_FSTUWT,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,PV6MATH,PV7MATH,PV8MATH,PV9MATH,PV10MATH,SCHLTYPE,PROATCE,STAFFSHORT,EDUSHORT,CLSIZE
0,EST,23300149.0,0.9668,4.69717,527.654,590.326,563.005,549.312,546.229,590.066,553.731,521.827,511.495,562.517,3.0,0.8667,NaN,-0.6483,18.0
1,EST,23300171.0,-0.3947,1.72897,715.893,719.525,738.151,742.997,732.032,697.641,687.527,695.576,679.676,677.391,2.0,0.7877,NaN,-0.9490,18.0
2,EST,23300075.0,-0.9071,1.76996,433.490,420.050,406.021,428.302,405.179,448.518,451.096,426.500,396.945,375.990,3.0,0.9727,NaN,1.2435,23.0
3,EST,23300194.0,-0.6361,2.48957,472.419,475.092,468.743,458.075,457.784,459.648,466.145,451.897,521.308,471.346,3.0,1.0000,NaN,0.4591,23.0
4,EST,23300079.0,-0.9308,1.43473,449.444,413.698,399.089,416.500,380.016,452.793,426.281,417.983,460.049,416.704,3.0,0.6800,NaN,0.4591,23.0
